# RQ3_preprocessing.ipynb
This notebook loads the raw dataset, preprocesses it, trains models, and outputs the required table (CSV) and figure (PDF).


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

try:
    from xgboost import XGBClassifier
    xgb_available=True
except:
    xgb_available=False

DATA_PATH = "/kaggle/input/cybersecurity-attacks/cybersecurity_attacks.csv"  # adjust if needed
df = pd.read_csv(DATA_PATH)

target = "Attack Type"

X = df.drop(columns=[target])
y = df[target]

X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


strategies = {
"Raw": X_train.fillna(0),
"Imputed": X_train.fillna(X_train.mean()),
"Scaled": X_train_scaled
}

scores=[]

for name,data in strategies.items():

    if name=="Scaled":
        model=RandomForestClassifier()
        model.fit(data,y_train)
        preds=model.predict(X_test_scaled)
    else:
        model=RandomForestClassifier()
        model.fit(data,y_train)
        preds=model.predict(X_test)

    acc=accuracy_score(y_test,preds)
    scores.append([name,acc])

table=pd.DataFrame(scores,columns=["Strategy","Accuracy"])
table.to_csv("RQ3_preprocessing_results.csv",index=False)

table.plot(x="Strategy",y="Accuracy",kind="bar")
plt.title("Preprocessing Impact")
plt.tight_layout()
plt.savefig("RQ3_preprocessing.pdf")
plt.show()
